# GateIO Data Downloader

### Objective:
- Download and save historical price data from the GateIO cryptocurrency exchange as CSV files.

In [5]:
# Collecting all imports in one place
import pandas as pd
import requests
import time
import random
import re
import os
import gzip
import shutil

## Function for data downloading

In [8]:
#Function for downloading data from GateIO website
def download_file(market_type='spot',      #Options: spot, future_usdt, future_btc
                  data_type='klines',      #For spot: klines, deals. For future_: klines, trades.
                  timeframe=None,          #For klines: 1m, 5m, 15m, 1h, 8h, 12h, 1d.
                  date='202401',           #Format: 202008
                  ticker='BTC_USDT'):
    
    #Format the final URL string if the data type is candlesticks
    if 'klines' in data_type.lower():
        url = f"https://download.gatedata.org/{market_type}/candlesticks_{timeframe}/{date}/{ticker}-{date}.csv.gz"
    #In all other cases
    else:
        url = f"https://download.gatedata.org/{market_type}/{data_type}/{date}/{ticker}-{date}.csv.gz"
    print(url)
    
    #Directory for saving the file
    directory = f"/Users/danielschollenberg/Documents/Трейдинг/Данные/gateio/{market_type}/{data_type}/{ticker}"
    #Create directory if it does not exist
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory {directory} created.")
            
    #Bypassing potential blocking:
    #Add random delay from 0 to 2 seconds before request
    delay = random.uniform(0, 2)
    print(f"Delay before request: {delay:.2f} seconds")
    time.sleep(delay)
    #List of User-Agent headers
    user_agents = ['Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
                   'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0.1 Safari/605.1.15',
                   'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:89.0) Gecko/20100101 Firefox/89.0',
                   'Mozilla/5.0 (iPhone; CPU iPhone OS 14_6 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Mobile/15E148 Safari/604.1',
                   'Mozilla/5.0 (Linux; Android 10; SM-G975F) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.105 Mobile Safari/537.36']
    #Select random User-Agent header
    headers = {'User-Agent': random.choice(user_agents)}
    
    #Create request to get data in streaming mode
    response = requests.get(url, headers=headers, stream=True)
    #If connection successful (status 200), download and save the file
    if response.status_code == 200:
        filename = directory + f"/{ticker}_{date}.zip"
        with open(filename, 'wb') as f:
            #Write all content directly to the file
            f.write(response.content)  
        print(f"File {filename} successfully downloaded and saved")
    else:
        print(f"Error downloading file: status {response.status_code}, URL: {url}")
        
    #Function for unpacking gz file
    def unpack_gz(gz_path, output_filename):
        with gzip.open(gz_path, 'rb') as f_in:
            with open(output_filename, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        #Delete gz file after extraction
        os.remove(gz_path)
        print(f"Archive {gz_path} deleted after extraction.")
    
    try:
        #Unpack gz file
        unpack_gz(filename, os.path.join(directory, f"{ticker}_{date}.csv"))
        print(f"File {filename} successfully extracted.")
    except:
        print('Error during extraction')

In [44]:
# Example of usage
download_file(data_type='deals', date='202401')

https://download.gatedata.org/spot/deals/202401/BTC_USDT-202401.csv.gz
Файл /Users/danielschollenberg/Documents/Трейдинг/Данные/gateio/spot/deals/BTC_USDT/BTC_USDT_202401.zip успешно скачан, сохранен на диск
Архив /Users/danielschollenberg/Documents/Трейдинг/Данные/gateio/spot/deals/BTC_USDT/BTC_USDT_202401.zip удален после распаковки.
Файл /Users/danielschollenberg/Documents/Трейдинг/Данные/gateio/spot/deals/BTC_USDT/BTC_USDT_202401.zip успешно распакован.
